# Pendulum tracking
Strategy: the pendulum is the only moving object, and it's darker than the backdrop. So we combine a **motion mask** (background subtraction) with a **darkness mask** (low HSV value). The intersection should isolate the pendulum bob, then we take the centroid of the largest contour per frame.

In [ ]:
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt

VIDEO = "video.mp4"

## 1. Look at a random frame to pick thresholds

In [ ]:
cap = cv2.VideoCapture(VIDEO)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
idx = random.randint(0, total - 1)
cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
ok, frame = cap.read()
cap.release()
print(f"total={total}  fps={fps}  random idx={idx}  shape={frame.shape}")
plt.figure(figsize=(9, 6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title(f"frame {idx}"); plt.axis('off'); plt.show()

## 2. Tune the darkness mask
HSV Value channel: 0 = black, 255 = white. Raise `V_MAX` until the pendulum is captured but most of the backdrop isn't. Also tweak `S_MIN` if the bob is colored vs. just dark.

In [ ]:
V_MAX = 90   # keep pixels darker than this
S_MIN = 0    # raise if you want colored (not just dark) pixels

hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
dark = cv2.inRange(hsv, (0, S_MIN, 0), (179, 255, V_MAX))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); ax[0].set_title('frame'); ax[0].axis('off')
ax[1].imshow(dark, cmap='gray'); ax[1].set_title(f'dark mask  V<{V_MAX}'); ax[1].axis('off')
plt.show()

## 3. Build the motion mask
MOG2 background subtractor, warmed up on the first N frames.

In [ ]:
WARMUP = 60
bg = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=25, detectShadows=False)

cap = cv2.VideoCapture(VIDEO)
for _ in range(WARMUP):
    ok, f = cap.read()
    if not ok: break
    bg.apply(f)

cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
ok, frame = cap.read()
cap.release()
motion = bg.apply(frame, learningRate=0)
motion = cv2.medianBlur(motion, 5)

plt.figure(figsize=(8, 5))
plt.imshow(motion, cmap='gray'); plt.title('motion mask (MOG2)'); plt.axis('off'); plt.show()

## 4. Combine masks and find the pendulum centroid

In [ ]:
def find_centroid(mask, min_area=50):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, mask
    c = max(contours, key=cv2.contourArea)
    if cv2.contourArea(c) < min_area:
        return None, mask
    M = cv2.moments(c)
    if M['m00'] == 0:
        return None, mask
    return (M['m10'] / M['m00'], M['m01'] / M['m00']), mask

combined = cv2.bitwise_and(dark, motion)
centroid, clean = find_centroid(combined)
print('centroid:', centroid)

disp = frame.copy()
if centroid is not None:
    cv2.circle(disp, (int(centroid[0]), int(centroid[1])), 15, (0, 0, 255), 3)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(clean, cmap='gray'); ax[0].set_title('dark AND motion (cleaned)'); ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(disp, cv2.COLOR_BGR2RGB)); ax[1].set_title('detected'); ax[1].axis('off')
plt.show()

## 5. Track across the whole video

In [ ]:
bg = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=25, detectShadows=False)
cap = cv2.VideoCapture(VIDEO)
xs, ys, ts = [], [], []
i = 0
while True:
    ok, f = cap.read()
    if not ok: break
    motion = bg.apply(f)
    if i < WARMUP:
        i += 1; continue
    motion = cv2.medianBlur(motion, 5)
    hsv = cv2.cvtColor(f, cv2.COLOR_BGR2HSV)
    dark = cv2.inRange(hsv, (0, S_MIN, 0), (179, 255, V_MAX))
    c, _ = find_centroid(cv2.bitwise_and(dark, motion))
    if c is not None:
        xs.append(c[0]); ys.append(c[1]); ts.append(i / fps)
    i += 1
cap.release()
print(f'tracked {len(xs)} / {total} frames')

In [ ]:
xs_a = np.array(xs)
ys_a = np.array(ys)
ts_a = np.array(ts)

# pivot estimate: center of horizontal swing range, and topmost y the bob reached
pivot_x = 0.5 * (xs_a.min() + xs_a.max())
pivot_y = ys_a.min()
print(f"pivot ≈ ({pivot_x:.1f}, {pivot_y:.1f}) px")

# signed angle from vertical (image y grows downward, so dy = y - pivot_y > 0 below pivot)
dx = xs_a - pivot_x
dy = ys_a - pivot_y
theta = np.arctan2(dx, dy)   # radians, 0 = straight down, + = right of pivot

plt.figure(figsize=(11, 4))
plt.plot(ts_a, np.degrees(theta), lw=1)
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('time (s)'); plt.ylabel('angle from vertical (deg)')
plt.title('pendulum amplitude')
plt.tight_layout(); plt.show()